# Transformer Decoder 与文本生成

## 学习目标

使用 causal mask 训练一个极简 decoder-only Transformer，并实现逐 token 生成。

## 概念模型

训练时输入序列右移一位作为目标；causal mask 保证位置不能看到未来 token。推理时每次把新 token 拼接到上下文末尾。

In [ ]:
import torch
from torch import nn
torch.manual_seed(42)
vocab, length, width = 12, 6, 16
class TinyDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(vocab, width)
        layer = nn.TransformerEncoderLayer(width, 2, batch_first=True)
        self.decoder = nn.TransformerEncoder(layer, 2)
        self.head = nn.Linear(width, vocab)
    def forward(self, tokens):
        n = tokens.size(1)
        h = self.embedding(tokens)
        mask = torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)
        return self.head(self.decoder(h, mask=mask))
model = TinyDecoder()
sequence = torch.randint(vocab, (8, length))
logits = model(sequence)
assert logits.shape == (8, length, vocab)
print(logits.shape)

### 实验 1：teacher forcing 训练

**实验目的**：输入序列去掉最后一个 token，目标去掉第一个 token，使每个位置预测下一个 token。logits 与 targets 展平后交给交叉熵。

causal mask 必须阻止位置看到未来 token，否则会发生标签泄漏。teacher forcing 训练看到真实历史，而生成时看到自身预测，两者存在 exposure bias。


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=0.03)
loss_fn = nn.CrossEntropyLoss()
for _ in range(3):
    optimizer.zero_grad(set_to_none=True)
    inputs, targets = sequence[:, :-1], sequence[:, 1:]
    loss = loss_fn(model(inputs).reshape(-1, vocab), targets.reshape(-1))
    loss.backward(); optimizer.step()
print('teacher-forcing loss:', loss.item())

### 实验 2：逐 token 自回归生成

**实验目的**：从 prefix 开始，每轮取最后位置 logits 的 argmax 作为下一个 token，并拼回输入。`eval()` 与 inference mode 保证确定性推理和低开销。

贪心解码不是唯一策略；采样、temperature、top-k/top-p 会改变多样性。还应处理 EOS、最大长度、位置编码上限和 KV cache。


In [ ]:
def generate(model, prefix, steps):
    model.eval(); result = prefix.clone()
    with torch.inference_mode():
        for _ in range(steps):
            next_token = model(result)[:, -1].argmax(dim=-1, keepdim=True)
            result = torch.cat([result, next_token], dim=1)
    return result
generated = generate(model, sequence[:1, :2], 3)
print('generated:', generated.tolist())
assert generated.shape == (1, 5)

## 检查点

解释 teacher forcing 与生成时输入的区别，说明 causal mask 的上三角为什么必须屏蔽。

## 试一试

将 argmax 改成 temperature sampling，并限制最大上下文长度。

## 常见错误与调试

目标没有右移、训练时看到未来 token、mask device 不一致、把序列维和 batch 维混淆、生成时忘记只取最后位置。